**My Personal Finance Buddy**

**Step 1 : Data collection & Store embeddings in ChromaDB**
- Collect transcripts from YouTube: download audio via yt-dlp, then transcribe with Whisper
- Use task="transcribe" when lang="en"
- Use task="translate" otherwise (e.g., FR → EN)
- Preprocess & chunk with RecursiveCharacterTextSplitter into ~500 characters with 100 overlap
- Embed & store the chunks in ChromaDB using OpenAIEmbeddings

In [7]:
# ----------------- Cell 1: YouTube Processing -----------------
import os
import re
import whisper
from langchain.vectorstores import Chroma
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain.schema import Document

video_map = {
    "ouvbeb2wSGA": {"name": "General_Literacy", "lang": "en"},
    "bb6_M_srMBk": {"name": "Investment", "lang": "en"},
    "MabD5R8kRak": {"name": "Financial_Goal", "lang": "en"},
    "7XVmhedNiIQ": {"name": "Changing_Habits", "lang": "en"},
    "pyEgYI7S-r4": {"name": "Credit_and_Debts", "lang": "en"},
    "q6d4VAIWFPA": {"name": "Level_of_Financial_Liberty", "lang": "fr"},
}

OUTPUT_DIR = r"C:\Users\katyd\Downloads\FINAL_PROJECT_OUTPUT\YT_video"
CHROMA_DB_PATH = r"C:\Users\katyd\Downloads\FINAL_PROJECT_OUTPUT\chroma_db"

def download_and_transcribe(video_id, model_size="base"):
    video_info = video_map.get(video_id, {"name": video_id, "lang": "en"})
    name, lang = video_info["name"], video_info["lang"]

    audio_path = os.path.join(OUTPUT_DIR, f"{name}.mp3")
    transcript_path = os.path.join(OUTPUT_DIR, f"{name}.txt")
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    # Use cached transcript if exists
    if os.path.exists(transcript_path):
        with open(transcript_path, "r", encoding="utf-8") as f:
            return f.read()

    # Download audio if not already there
    if not os.path.exists(audio_path):
        cmd = f'yt-dlp -x --audio-format mp3 https://www.youtube.com/watch?v={video_id} -o "{audio_path}"'
        os.system(cmd)

    if not os.path.exists(audio_path):
        raise FileNotFoundError(f"❌ Failed: {audio_path} not created.")

    # Whisper transcription (translate FR → EN if needed)
    model = whisper.load_model(model_size)
    task = "transcribe" if lang == "en" else "translate"
    result = model.transcribe(audio_path, task=task, language=lang)

    with open(transcript_path, "w", encoding="utf-8") as f:
        f.write(result["text"])

    return result["text"]

def preprocess_and_chunk(transcripts, chunk_size=500, chunk_overlap=100):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, 
        chunk_overlap=chunk_overlap
    )
    docs = []
    for video_id, text in transcripts.items():
        cleaned = re.sub(r"\s+", " ", text).strip()
        doc = Document(page_content=cleaned, metadata={"source": video_id})
        docs.extend(splitter.split_documents([doc]))
    return docs

def create_or_load_vector_db(documents=None):
    embeddings = OpenAIEmbeddings()
    if os.path.exists(CHROMA_DB_PATH) and not documents:
        return Chroma(persist_directory=CHROMA_DB_PATH, embedding_function=embeddings)
    if documents:
        db = Chroma.from_documents(documents, embeddings, persist_directory=CHROMA_DB_PATH)
        db.persist()
        return db
    return None







**Step 2 : RAG (Ingestion + Vector Store + create_rag_chain)**
- Setup: Load .env; initialize GPT-4o (low temp), OpenAIEmbeddings, and persisted Chroma vector store.

- Retrieval: MultiQueryRetriever expands queries; CohereRerank re-ranks; ContextualCompressionRetriever returns the top, cleaner context.

- QA Chain: History-aware retriever + strict context-only prompt → answer generation via create_stuff_documents_chain

In [9]:
# ----------------- Cell 2: RAG Setup -----------------
from langchain_chroma import Chroma
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain.chains import create_history_aware_retriever, create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain

CHROMA_DB_PATH = r"C:\Users\katyd\Downloads\FINAL_PROJECT_OUTPUT\chroma_db"

def create_rag_chain(llm=None, vector_store=None):
    if not vector_store:
        embeddings = OpenAIEmbeddings()
        vector_store = Chroma(persist_directory=CHROMA_DB_PATH, embedding_function=embeddings)

    if not llm:
        llm = ChatOpenAI(model="gpt-4o", temperature=0.2)

    retriever = vector_store.as_retriever(search_kwargs={"k": 3})

    contextualize_prompt = ChatPromptTemplate.from_messages([
        ("system", "You are 'Coach', a financial assistant. Use conversation history to refine the query."),
        MessagesPlaceholder(variable_name="chat_history"),
        ("human", "{input}")
    ])
    history_aware_retriever = create_history_aware_retriever(llm, retriever, contextualize_prompt)

    qa_prompt = ChatPromptTemplate.from_messages([
        ("system", "You are 'Coach'. Use ONLY the following context to answer:\n{context}"),
        MessagesPlaceholder(variable_name="chat_history"),
        ("human", "{input}")
    ])
    doc_chain = create_stuff_documents_chain(llm, qa_prompt)

    return create_retrieval_chain(history_aware_retriever, doc_chain)

rag_chain = create_rag_chain()




**Step 3 : Agent (Tools + Flow)**
- Collects user financial profile step by step with plan templates.
- Evaluates financial health (Strong/Moderate/Weak) and gives advice.
- Enriches plans with live web data (e.g., mortgage rates).
- Powers a GPT-4o tools agent that uses the profile to guide coaching.

In [12]:
!pip install deep-translator


In [14]:
# ----------------- Cell 3: Agent Setup -----------------
from langchain_openai import ChatOpenAI
from langchain.prompts import ChatPromptTemplate
from langchain.agents import AgentExecutor, create_openai_tools_agent
from langchain_core.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun
import re
from langdetect import detect
from deep_translator import GoogleTranslator  # ✅ use deep-translator instead of googletrans

# --- Globals ---
user_profile = {}
plan_states = {}
last_plan = None
pending_followup = None

# default translator (auto-detect → English)
translator = GoogleTranslator(source="auto", target="en")

# --- Helpers ---
def detect_language(text):
    try:
        return detect(text)
    except:
        return "en"

def translate(text, lang):
    """
    Translate text to the target language using deep-translator.
    If target lang is 'en', returns the original.
    """
    if not text:
        return ""
    if lang == "en":
        return text
    try:
        return GoogleTranslator(source="auto", target=lang).translate(text)
    except:
        return text

def extract_number(text: str) -> str:
    if not text: return ""
    percent_match = re.search(r"\d+(\.\d+)?(?=%)", text)
    if percent_match: return percent_match.group()
    match = re.search(r"\d{1,3}(?:[.,]\d{3})*(?:[.,]\d+)?|\d+", text.replace(",", ""))
    return match.group().replace(",", "") if match else text.strip()

def evaluate_financial_health(state):
    try:
        income = float(state.get("income", 0))
        savings = float(state.get("savings", 0))
        expenses = float(state.get("expenses", 0))
        debt = float(state.get("debt", 0))
    except ValueError:
        return "⚠️ Could not evaluate financial health.", "Unknown"

    reasons, consequences = [], []
    savings_rate = savings / max(income, 1)
    debt_ratio = debt / max(income, 1)
    expense_ratio = expenses / max(income, 1)

    if savings_rate < 0.2:
        reasons.append("Low savings rate (<20% of income)")
        consequences.append("Harder to build an emergency fund or down payment.")
    if debt_ratio >= 0.5:
        reasons.append("High debt load (≥50% of income)")
        consequences.append("Limits borrowing capacity and increases financial risk.")
    if expense_ratio >= 0.6:
        reasons.append("High expenses (≥60% of income)")
        consequences.append("Leaves little room for saving and shocks.")

    if savings_rate >= 0.5 and debt_ratio < 0.2 and expense_ratio < 0.6:
        return (
            "💎 Strong – Excellent! You are financially strong. Focus on investments and long-term goals.",
            "Strong",
        )
    elif savings_rate >= 0.2 and debt_ratio < 0.5:
        return (
            f"📈 Moderate – On the right track. Issues: {', '.join(reasons) or 'None'}. "
            "Tip: boost savings rate for resilience.",
            "Moderate",
        )
    else:
        return (
            f"⚠️ Weak – Vulnerable finances. Issues: {', '.join(reasons) or 'General instability'}. "
            f"Consequences: {', '.join(consequences) or 'Difficulty reaching financial goals.'}",
            "Weak",
        )

# --- Tool: Create Plan ---
@tool("create_plan", return_direct=False)
def create_plan(plan_type: str, user_input: str) -> str:
    """
    Build personalized financial plans (house, trip, savings).
    - Ask one missing field at a time.
    - Summarize only when profile complete.
    - Include health level with explanation.
    - Auto-suggest savings plan if Weak or <20% down payment.
    """
    global last_plan, pending_followup
    plan_type = plan_type.lower().strip()

    # Schemas
    schemas = {
        "house": {
            "required": ["price", "savings", "income", "expenses", "debt", "location", "timeframe", "interest_rate"],
            "questions": {
                "price": "🏠 What is the estimated price of the house?",
                "savings": "💰 How much savings do you currently have?",
                "income": "💵 What is your monthly income?",
                "expenses": "📊 What are your monthly expenses?",
                "debt": "💳 Do you currently have any debts?",
                "location": "📍 Where would you like to buy the house?",
                "timeframe": "⏳ What is your target timeline (e.g., 2 years)?",
                "interest_rate": "📈 What mortgage interest rate are you expecting?",
            },
        },
        "savings": {
            "required": ["income", "expenses", "savings", "goal", "timeframe"],
            "questions": {
                "income": "💵 What is your monthly income?",
                "expenses": "📊 What are your monthly expenses?",
                "savings": "💰 How much savings do you already have?",
                "goal": "🎯 What is your savings goal?",
                "timeframe": "📅 What is your target timeframe?",
            },
        },
        "trip": {
            "required": ["destination", "dates", "budget", "preferences"],
            "questions": {
                "destination": "🌍 Where would you like to travel?",
                "dates": "📅 When do you plan to travel?",
                "budget": "💵 What is your total budget?",
                "preferences": "🎯 Any travel preferences (hotels, activities, flights)?",
            },
        },
    }

    if plan_type not in schemas:
        return f"⚠️ I don’t yet support {plan_type} plans."

    schema = schemas[plan_type]
    if plan_type not in plan_states:
        plan_states[plan_type] = {"answers": {}, "last_field": None}

    state = plan_states[plan_type]["answers"]
    last_field = plan_states[plan_type]["last_field"]

    # Store last answer
    if last_field and last_field not in state:
        state[last_field] = extract_number(user_input)
        user_profile[last_field] = state[last_field]
        plan_states[plan_type]["last_field"] = None

    # Find missing
    missing = [f for f in schema["required"] if f not in state]
    if missing:
        next_field = missing[0]

        # Auto mortgage rate
        if next_field == "interest_rate":
            try:
                query = f"current average mortgage interest rate in {state.get('location','Europe')} 2025"
                result = DuckDuckGoSearchRun().run(query)
                rate_match = re.search(r"\d+(\.\d+)?%", result)
                if rate_match:
                    avg_rate = rate_match.group()
                    return f"I found an average mortgage rate: {avg_rate}. Would you like me to use this?"
            except:
                pass

        plan_states[plan_type]["last_field"] = next_field
        return schema["questions"][next_field]

    # Completed
    advice, level = evaluate_financial_health(state)
    user_profile["Project"] = plan_type.capitalize()
    user_profile["Financial Health"] = level

    summary = "\n".join([f"- {k.capitalize()}: {v}" for k, v in state.items()])
    last_plan = f"""✅ Your {plan_type.capitalize()} Plan

**Profile**
{summary}

**Financial Health**
{advice}

**Roadmap**
1. Save strategically.
2. Manage expenses.
3. Stay consistent with your timeline.
"""

    # Suggest savings plan if needed
    if plan_type == "house":
        savings_needed = 0.2 * float(state.get("price", 0))
        if level in ["Weak", "Moderate"] or float(state.get("savings", 0)) < savings_needed:
            return last_plan + "\n👉 Would you like me to create a savings plan to help you reach your down payment goal?"

    return last_plan

# --- Web Search Tool ---
@tool("web_search")
def web_search(query: str) -> str:
    """Search the web for financial information such as current rates, tips, or news."""
    return DuckDuckGoSearchRun().run(query)

# --- Agent Setup ---
llm = ChatOpenAI(model="gpt-4o", temperature=0.2)
tools = [create_plan, web_search]

agent_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a financial coach. Ask one question at a time. Summarize profile only when complete. Explain financial health like a teacher."),
    ("placeholder", "{chat_history}"),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}"),
])

agent = create_openai_tools_agent(llm, tools, agent_prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)











**STEP 4 : EVALUATION**


In [19]:
# ===================== Cell 4: Evaluation =====================
import os
from dotenv import load_dotenv
from langsmith import Client
from langchain_openai import ChatOpenAI
from langchain.prompts import ChatPromptTemplate
from langchain.smith import RunEvalConfig
from langchain.evaluation import EvaluatorType

# ----------------- Setup -----------------
load_dotenv()
os.environ["LANGCHAIN_TRACING_V2"] = "true"
# Ensure LANGCHAIN_API_KEY is set in your .env file
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAIN_API_KEY")
PROJECT_NAME = os.getenv("LANGCHAIN_PROJECT") or "Personal-Finance-Buddy-Evaluation"

client = Client()

# ----------------- Chatbot Wrapper -----------------
def run_chatbot_qa(example, chat_history: list) -> dict:
    """
    Wrapper for the chatbot pipeline.
    Accepts either dataset rows (with 'question' or 'input') or raw strings.
    """
    try:
        if isinstance(example, dict):
            # Handle both dataset schema ('question') and custom schema ('input')
            user_query = example.get("input") or example.get("question")
        else:
            # Raw string provided
            user_query = str(example)

        result = agent_executor.invoke({"input": user_query, "chat_history": chat_history})
        return {"answer": result["output"]}
    except Exception as e:
        return {"answer": f"⚠️ Error: {str(e)}"}

# ----------------- Self-Test -----------------
print("\n--- Running quick self-test ---")

self_test_cases = [
    {"q": "What is a budget?", "expected": ["budget", "income", "expenses"]},
    {"q": "What's the principle of forced saving?", "expected": ["saving", "automatically"]},
    {"q": "What's the difference between an emergency fund and investment savings?", "expected": ["emergency", "investment"]},
    {"q": "What's more important, paying off debt or saving?", "expected": ["debt", "saving"]},
    {"q": "If I save $200 per month for 5 years at 5% annual interest, how much will I have?", "expected": ["$", "13"]},
    {"q": "Should I invest in real estate right now?", "expected": ["cannot", "advice"]},
    {"q": "If I only pay the minimum on my credit card, I won’t pay interest, right?", "expected": ["interest", "debt"]},
    {"q": "What does compound interest mean in simple terms?", "expected": ["interest", "grows", "time"]},
]

for case in self_test_cases:
    resp = run_chatbot_qa(case["q"], chat_history=[])
    answer = resp["answer"]
    print(f"\nQ: {case['q']}\nA: {answer}")
    missing = [w for w in case["expected"] if w.lower() not in answer.lower()]
    print("✅ Passed\n" if not missing else f"❌ Missing: {missing}\n")

# ----------------- Multi-turn Plan Scenario -----------------
print("\n--- Running multi-turn scenario test ---")
scenario_test = {
    "q": "Help me to buy a house",
    "steps": ["My income is 80,000", "I have 20,000 in savings", "Expenses are 2000 per month", "No debts", "House price is 250,000", "In Porto"],
    "expected": ["income", "savings", "expenses", "debt", "price", "location"],
}

chat_history = []
resp = run_chatbot_qa(scenario_test["q"], chat_history)
print(f"\nUser: {scenario_test['q']}\nBot: {resp['answer']}")
chat_history.append((scenario_test["q"], resp["answer"]))

for step in scenario_test["steps"]:
    resp = run_chatbot_qa(step, chat_history)
    print(f"User: {step}\nBot: {resp['answer']}")
    chat_history.append((step, resp["answer"]))

final_answer = resp["answer"]
missing = [w for w in scenario_test["expected"] if w.lower() not in final_answer.lower()]
print("\n✅ Scenario Passed\n" if not missing else f"❌ Scenario missing: {missing}\n")

# ----------------- Complex Multi-turn Scenario -----------------
print("\n--- Running complex multi-turn scenario test ---")
complex_scenario_test = {
    "q": "I'm worried about my financial future. What should I do?",
    "steps": [
        "I'm not sure if I should save more or pay off my credit card debt first. I owe about $5,000.",
        "I also have some old student loans, but I only make a small payment each month, so I'm not worried about them.",
        "My salary is $60,000, but last month I told you it was $50,000. Sorry, I was just guessing before."
    ],
    "expected": ["future", "pay off", "credit card", "student loan", "salary"],
}

chat_history = []
resp = run_chatbot_qa(complex_scenario_test["q"], chat_history)
print(f"\nUser: {complex_scenario_test['q']}\nBot: {resp['answer']}")
chat_history.append((complex_scenario_test["q"], resp["answer"]))

for step in complex_scenario_test["steps"]:
    resp = run_chatbot_qa(step, chat_history)
    print(f"User: {step}\nBot: {resp['answer']}")
    chat_history.append((step, resp["answer"]))

final_answer = resp["answer"]
missing = [w for w in complex_scenario_test["expected"] if w.lower() not in final_answer.lower()]
print("\n✅ Complex Scenario Passed\n" if not missing else f"❌ Complex Scenario missing: {missing}\n")

# ----------------- Evaluation Config -----------------
evaluation_config = RunEvalConfig(
    evaluators=[
        RunEvalConfig.LabeledScoreString(
            evaluator_type=EvaluatorType.LABELED_CRITERIA,
            criteria={
                "correctness": "Does the answer match the ground truth?",
                "clarity": "Is the answer clear and easy to understand?",
                "helpfulness": "Is the answer useful and actionable for a beginner?",
            },
        )
    ]
)

# ----------------- Run Evaluation -----------------
print("\n--- Running full evaluation on dataset ---")
run_results = client.run_on_dataset(
    dataset_name="Financial-Coach-Complex-Tests",
    llm_or_chain_factory=lambda x: run_chatbot_qa(x, chat_history=[]),
    evaluation=evaluation_config,
    verbose=True,
)

print("\n✅ Evaluation complete! Check LangSmith dashboard.")








--- Running quick self-test ---


> Entering new AgentExecutor chain...
A budget is a financial plan that outlines expected income and expenses over a specific period, typically a month or a year. It helps individuals or organizations manage their money by tracking how much they earn, how much they spend, and how much they save. A budget can help you:

1. **Control Spending**: By setting limits on how much you can spend in different categories, such as groceries, entertainment, and housing.
2. **Achieve Financial Goals**: By allocating funds towards savings, investments, or paying off debt.
3. **Prepare for Emergencies**: By setting aside money in an emergency fund.
4. **Make Informed Financial Decisions**: By understanding your financial situation and making adjustments as needed.

Would you like to know how to create a budget or need help with a specific financial plan?

> Finished chain.

Q: What is a budget?
A: A budget is a financial plan that outlines expected income and expense

,feedback.correctness-clarity-helpfulness,error,execution_time,run_id
count,5.000000,0,5.000000,5
unique,NaN,0,NaN,5
top,NaN,NaN,NaN,19ea7a2c-a067-4a4f-acae-dfc0770c4041
freq,NaN,NaN,NaN,1
mean,0.600000,NaN,3.708742,NaN
std,0.547723,NaN,3.346307,NaN
min,0.000000,NaN,1.585791,NaN
25%,0.000000,NaN,2.203473,NaN
50%,1.000000,NaN,2.228234,NaN
75%,1.000000,NaN,2.888430,NaN



✅ Evaluation complete! Check LangSmith dashboard.


**GRADIO**

In [20]:
!pip install gradio
!pip install fpdf2

  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
Using cached httpcore-1.0.9-py3-none-any.whl (78 kB)
Using cached h11-0.16.0-py3-none-any.whl (37 kB)

  Attempting uninstall: h11

    Found existing installation: h11 0.9.0

    Uninstalling h11-0.9.0:

      Successfully uninstalled h11-0.9.0

   ---------------------------------------- 0/3 [h11]
   ---------------------------------------- 0/3 [h11]
  Attempting uninstall: httpcore
   ---------------------------------------- 0/3 [h11]
    Found existing installation: httpcore 0.9.1
   ---------------------------------------- 0/3 [h11]
   ------------- -------------------------- 1/3 [httpcore]
    Uninstalling httpcore-0.9.1:
   ------------- -------------------------- 1/3 [httpcore]
      Successfully uninstalled httpcore-0.9.1
   ------------- -------------------------- 1/3 [httpcore]
   ------------- -------------------------- 1/3 [httpcore]
   -------------

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
googletrans 4.0.0rc1 requires httpx==0.13.3, but you have httpx 0.28.1 which is incompatible.


In [ ]:
# ----------------- Cell 5: Gradio UI -----------------
import gradio as gr
import tempfile
from fpdf import FPDF
import os
import speech_recognition as sr
import pandas as pd

# ---------- Utilities ----------
def speech_to_text(audio_file):
    recognizer = sr.Recognizer()
    with sr.AudioFile(audio_file) as source:
        audio = recognizer.record(source)
    try:
        return recognizer.recognize_google(audio)
    except:
        return "⚠️ Speech recognition error."

def save_conversation_as_pdf(history, profile):
    pdf = FPDF()
    pdf.add_page()
    pdf.set_font("Arial", size=12)
    pdf.cell(200, 10, "Conversation Export", ln=True, align="C")
    pdf.ln(10)

    pdf.multi_cell(0, 10, "**User Profile**")
    for k, v in profile.items():
        pdf.multi_cell(0, 10, f"- {k}: {v}")

    pdf.ln(5)
    pdf.multi_cell(0, 10, "**Conversation**")
    for msg in history:
        pdf.multi_cell(0, 10, f"{msg['role']}: {msg['content']}")

    path = os.path.join(tempfile.gettempdir(), "conversation.pdf")
    pdf.output(path)
    return path

def save_conversation_as_excel(profile, health, plan):
    profile_df = pd.DataFrame(list(profile.items()), columns=["Profile Key", "Value"])
    health_df = pd.DataFrame(list(health.items()), columns=["Metric", "Evaluation"])
    plan_df = pd.DataFrame(list(plan.items()), columns=["Step", "Action"])

    path = os.path.join(tempfile.gettempdir(), "financial_report.xlsx")
    with pd.ExcelWriter(path, engine="openpyxl") as writer:
        profile_df.to_excel(writer, sheet_name="User Profile", index=False)
        health_df.to_excel(writer, sheet_name="Financial Health", index=False)
        plan_df.to_excel(writer, sheet_name="Action Plan", index=False)
    return path

# ---------- Chatbot State ----------
chatbot_history = []

def chatbot_interface(message, audio):
    global user_profile, chatbot_history
    if audio:
        message = speech_to_text(audio)

    result = agent_executor.invoke({
        "input": message,
        "chat_history": [(m["role"], m["content"]) for m in chatbot_history]
    })
    final_answer = result["output"]

    chatbot_history.append({"role": "user", "content": message})
    chatbot_history.append({"role": "assistant", "content": final_answer})
    return chatbot_history, user_profile

def reset_profile():
    global user_profile, plan_states, chatbot_history
    user_profile, plan_states, chatbot_history = {}, {}, []
    return [], {}

# ---------- Gradio UI ----------
with gr.Blocks() as demo:
    gr.Markdown("## 🤖 Personal Finance & Planning Assistant")

    with gr.Row():
        msg = gr.Textbox(label="💬 Your message")
        audio = gr.Audio(sources=["microphone"], type="filepath", label="🎤 Speak")

    with gr.Row():
        send_btn = gr.Button("Send")
        reset_btn = gr.Button("🔄 Reset Profile")
        save_pdf_btn = gr.Button("💾 Save Conversation as PDF")
        save_excel_btn = gr.Button("📊 Save Report as Excel")

    chatbot = gr.Chatbot(type="messages", label="Conversation")
    profile_display = gr.JSON(label="📂 Current User Profile")
    pdf_output = gr.File(label="📄 Download PDF")
    excel_output = gr.File(label="📊 Download Excel")

    # Button actions
    send_btn.click(chatbot_interface, [msg, audio], [chatbot, profile_display])
    reset_btn.click(reset_profile, None, [chatbot, profile_display])
    save_pdf_btn.click(lambda: save_conversation_as_pdf(chatbot_history, user_profile), None, pdf_output)
    save_excel_btn.click(
        lambda: save_conversation_as_excel(
            user_profile,
            evaluate_financial_health(user_profile),
            create_plan(user_profile)
        ),
        None,
        excel_output
    )

demo.launch(share=True)






* Running on local URL:  http://127.0.0.1:7860

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.




> Entering new AgentExecutor chain...
To help you create a personalized financial plan for buying a house in Italy, I need to know a few more details:

1. What is your current monthly expense?
2. Do you have any other debts or financial obligations?
3. Are you planning to make a down payment, and if so, how much?
4. What is your preferred timeline for purchasing the house?

> Finished chain.


> Entering new AgentExecutor chain...
To create a personalized financial plan for buying a house, I need to know:

1. What is your current monthly expense?
2. Do you have any other debts or financial obligations?
3. Are you planning to make a down payment, and if so, how much?
4. What is your preferred timeline for purchasing the house?

> Finished chain.


> Entering new AgentExecutor chain...
How much have you saved for the down payment on the house?

> Finished chain.


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "c:\Users\katyd\anaconda3\envs\mon_env\Lib\site-packages\uvicorn\protocols\http\httptools_impl.py", line 409, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\katyd\anaconda3\envs\mon_env\Lib\site-packages\uvicorn\middleware\proxy_headers.py", line 60, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\katyd\anaconda3\envs\mon_env\Lib\site-packages\fastapi\applications.py", line 1054, in __call__
    await super().__call__(scope, receive, send)
  File "c:\Users\katyd\anaconda3\envs\mon_env\Lib\site-packages\starlette\applications.py", line 113, in __call__
    await self.middleware_stack(scope, receive, send)
  File "c:\Users\katyd\anaconda3\envs\mon_env\Lib\site-packages\starlette\middleware\errors.py", line 186, in __call__
    rais



> Entering new AgentExecutor chain...
A down payment is an upfront payment you make when purchasing a large asset, like a house. It's usually a percentage of the purchase price. For a house, the down payment can range from 5% to 20% or more of the home's price. The larger the down payment, the less you need to borrow, which can lead to lower monthly mortgage payments and potentially better loan terms.

For a 300,000 euros house, a 20% down payment would be 60,000 euros. How much have you saved for this purpose?

> Finished chain.


> Entering new AgentExecutor chain...
A down payment is an initial payment made when buying something on credit, such as a house. It is typically expressed as a percentage of the total purchase price. For example, if you're buying a house for 300,000 euros, a 20% down payment would be 60,000 euros. This amount is paid upfront, and the rest is usually financed through a mortgage. 

How much have you saved for the down payment on your house?

> Finished chai

ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "c:\Users\katyd\anaconda3\envs\mon_env\Lib\site-packages\uvicorn\protocols\http\httptools_impl.py", line 409, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\katyd\anaconda3\envs\mon_env\Lib\site-packages\uvicorn\middleware\proxy_headers.py", line 60, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\katyd\anaconda3\envs\mon_env\Lib\site-packages\fastapi\applications.py", line 1054, in __call__
    await super().__call__(scope, receive, send)
  File "c:\Users\katyd\anaconda3\envs\mon_env\Lib\site-packages\starlette\applications.py", line 113, in __call__
    await self.middleware_stack(scope, receive, send)
  File "c:\Users\katyd\anaconda3\envs\mon_env\Lib\site-packages\starlette\middleware\errors.py", line 186, in __call__
    rais



> Entering new AgentExecutor chain...
What is the loan term you are considering for your mortgage (e.g., 15 years, 30 years)?

> Finished chain.


> Entering new AgentExecutor chain...
What is the loan term you are considering for your mortgage (e.g., 15 years, 30 years)?

> Finished chain.


> Entering new AgentExecutor chain...
What is the loan term you are considering for your mortgage (e.g., 15 years, 30 years)?

> Finished chain.


> Entering new AgentExecutor chain...
Do you know the current interest rate for a mortgage in Italy, or would you like me to find that information for you?

> Finished chain.


> Entering new AgentExecutor chain...
¿Cuál es el precio del coche que te interesa comprar?

> Finished chain.
